In [60]:
import pandas as pd 

data_tags=pd.read_csv('ml-latest-small/tags.csv')
data_tags.head(20)

,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200
5,2,89774,Tom Hardy,1445715205
6,2,106782,drugs,1445715054
7,2,106782,Leonardo DiCaprio,1445715051
8,2,106782,Martin Scorsese,1445715056
9,7,48516,way too long,1169687325


In [61]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf=TfidfVectorizer(stop_words='english')

data_tags['tag']=data_tags['tag'].fillna('')

tfidf_matrix=tfidf.fit_transform(data_tags['tag'])
tfidf_matrix.shape


(3683, 1673)

In [62]:
tfidf.get_feature_names_out()[10:20]

array(['2d', '70mm', '80', 'aardman', 'abortion', 'absorbing', 'abstract',
       'abuse', 'academy', 'accident'], dtype=object)

In [63]:
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
cosine_sim.shape


(3683, 3683)

In [64]:
cosine_sim[1]

array([0., 1., 0., ..., 0., 0., 0.], shape=(3683,))

In [65]:
data_movies=pd.read_csv('ml-latest-small/movies.csv')
metadata=pd.merge(data_tags,data_movies,on="movieId")
metadata.head(20)

,userId,movieId,tag,timestamp,title,genres
0,2,60756,funny,1445714994,Step Brothers (2008),Comedy
1,2,60756,Highly quotable,1445714996,Step Brothers (2008),Comedy
2,2,60756,will ferrell,1445714992,Step Brothers (2008),Comedy
3,2,89774,Boxing story,1445715207,Warrior (2011),Drama
4,2,89774,MMA,1445715200,Warrior (2011),Drama
5,2,89774,Tom Hardy,1445715205,Warrior (2011),Drama
6,2,106782,drugs,1445715054,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama
7,2,106782,Leonardo DiCaprio,1445715051,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama
8,2,106782,Martin Scorsese,1445715056,"Wolf of Wall Street, The (2013)",Comedy|Crime|Drama
9,7,48516,way too long,1169687325,"Departed, The (2006)",Crime|Drama|Thriller


In [66]:
metadata['genres'] = metadata['genres'].str.replace('|', ' ', regex=False)

In [67]:
indices = pd.Series(metadata.index, index=metadata['title']).drop_duplicates()
indices[:10]

title
Step Brothers (2008)               0
Step Brothers (2008)               1
Step Brothers (2008)               2
Warrior (2011)                     3
Warrior (2011)                     4
Warrior (2011)                     5
Wolf of Wall Street, The (2013)    6
Wolf of Wall Street, The (2013)    7
Wolf of Wall Street, The (2013)    8
Departed, The (2006)               9
dtype: int64

In [68]:
metadata['tag'] = metadata['tag'].fillna('')
metadata['soup'] = metadata['tag'].astype(str) + ' ' + metadata['genres'].astype(str)

In [69]:
metadata['genres'] = metadata['genres'].str.replace('|', ' ', regex=False)

In [72]:
# sažimanje tagova za isti film u jedan dugački tekst
metadata_clean = metadata.groupby('title')['soup'].apply(lambda x: ' '.join(x)).reset_index()
metadata_clean.head(20)


,title,soup
0,(500) Days of Summer (2009),artistic Comedy Drama Romance Funny Comedy Dra...
1,...And Justice for All (1979),lawyers Drama Thriller
2,10 Cloverfield Lane (2016),creepy Thriller suspense Thriller
3,10 Things I Hate About You (1999),Shakespeare sort of Comedy Romance
4,101 Dalmatians (1996),dogs Adventure Children Comedy remake Adventur...
5,101 Dalmatians (One Hundred and One Dalmatians...,Disney Adventure Animation Children
6,"11'09""01 - September 11 (2002)",terrorism Drama
7,12 Angry Men (1957),court Drama claustrophobic Drama confrontation...
8,127 Hours (2010),stranded Adventure Drama Thriller
9,13 Going on 30 (2004),Mark Ruffalo Comedy Fantasy Romance


In [75]:
test_film = metadata[metadata['title'] == '3:10 to Yuma (2007)']
print(test_film[['userId', 'tag', 'genres']])

     userId             tag                      genres
678     357  Christian Bale  Action Crime Drama Western
